# 02_select_best_model 

In this notebook, we will train different type of model (SVM, Random Forest and linear regression), evaluate their performances, then select the best to compare it with chatGPT performances. 


**Inputs**

The complete dataset in csv format, constructed in the previous notebook. 


**Outputs**

The different models and their evaluation. 

## 1. Library import 

In [1]:

# Data Processing
import pandas as pd
import joblib

# Modelling
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm

from sklearn.metrics import accuracy_score
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

from scipy.sparse import hstack
from scipy.sparse import csr_matrix


## 2. Data import 

We first import the datasets we created in the previous notebook 

In [2]:
df = pd.read_csv("metrics//data//data_fanfic_jk.csv")
df = df.sample(frac =1.0)
df

,text,paragraph,categorie,nombre_mots,tokens,pos_tags,function_word_freq,type_token_ratio,avg_word_length,noun_ratio,verb_ratio,adj_ratio,adv_ratio,exclamation_freq,question_freq,comma_freq,avg_sentence_length
5840,Running on Air - eleventy7.txt,"Yes. Harry remembers that quiet moment, buried...",0,107,"['Yes', '.', 'Harry', 'remembers', 'that', 'qu...","['INTJ', 'PUNCT', 'PROPN', 'VERB', 'SCONJ', 'A...",0.459016,0.631148,3.942623,0.155738,0.139344,0.049180,0.032787,0.0,0.000000,0.057377,17.428571
3498,Harry Potter and the Shadowed Light - Writingb...,“The first thing I did was look myself up. My ...,0,121,"['“', 'The', 'first', 'thing', 'I', 'did', 'wa...","['PUNCT', 'DET', 'ADJ', 'NOUN', 'PRON', 'VERB'...",0.546763,0.625899,3.870504,0.194245,0.136691,0.050360,0.043165,0.0,0.000000,0.043165,17.375000
1021,"BLOODY, SLUTTY, AND PATHETIC - WhatMurdah.txt","But, Hermione was beginning to understand, Luc...",0,138,"['But', ',', 'Hermione', 'was', 'beginning', '...","['CCONJ', 'PUNCT', 'PROPN', 'AUX', 'VERB', 'PA...",0.480000,0.594286,4.177143,0.097143,0.108571,0.051429,0.062857,0.0,0.000000,0.040000,21.875000
1964,Burning Red - NoNameWriter.txt,The clear answer was to work at getting better...,0,113,"['The', 'clear', 'answer', 'was', 'to', 'work'...","['DET', 'ADJ', 'NOUN', 'AUX', 'PART', 'VERB', ...",0.564516,0.717742,3.887097,0.185484,0.177419,0.048387,0.032258,0.0,0.000000,0.032258,41.333333
7243,The Cadence of Part-time Poets - motswolo.txt,Before he could finish Peter was already tippi...,0,115,"['Before', 'he', 'could', 'finish', 'Peter', '...","['SCONJ', 'PRON', 'AUX', 'VERB', 'PROPN', 'AUX...",0.523810,0.642857,4.103175,0.158730,0.174603,0.031746,0.079365,0.0,0.000000,0.031746,25.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3525,Harry Potter and the Shadowed Light - Writingb...,Sirius stood shocked as the soul fragment from...,0,111,"['Sirius', 'stood', 'shocked', 'as', 'the', 's...","['PROPN', 'VERB', 'VERB', 'ADP', 'DET', 'NOUN'...",0.556452,0.677419,3.983871,0.088710,0.137097,0.096774,0.072581,0.0,0.008065,0.040323,20.666667
4766,Lily's Boy - SomewheresSword.txt,"The spring term began to earn its name, as Jan...",0,143,"['The', 'spring', 'term', 'began', 'to', 'earn...","['DET', 'NOUN', 'NOUN', 'VERB', 'PART', 'VERB'...",0.475000,0.650000,4.293750,0.175000,0.125000,0.100000,0.062500,0.0,0.000000,0.056250,40.000000
5321,On Punching Gods and Absentee Dads - Enigmaris...,“I am sorry. I didn’t even think about how jar...,0,113,"['“', 'I', 'am', 'sorry', '.', 'I', 'did', 'n’...","['PUNCT', 'PRON', 'AUX', 'ADJ', 'PUNCT', 'PRON...",0.529851,0.656716,3.589552,0.126866,0.141791,0.067164,0.044776,0.0,0.000000,0.044776,19.142857
356,All the Young Dudes - MsKingBean89.txt,Remus approached with caution. Was this how it...,0,128,"['Remus', 'approached', 'with', 'caution', '.'...","['PROPN', 'VERB', 'ADP', 'NOUN', 'PUNCT', 'AUX...",0.489655,0.641379,3.841379,0.137931,0.131034,0.068966,0.013793,0.0,0.027586,0.034483,13.181818


In [3]:
# Load vectorizer 
tfidf = joblib.load("metrics/data/tfidf_vectorizer.pkl")


## 3. Train test split 

We first split our data before train our models. 

In [4]:
# All the features we will use to train our models 

stylometric_columns = [
    "function_word_freq",
    "type_token_ratio",
    "avg_word_length",
    "noun_ratio",
    "verb_ratio",
    "adj_ratio",
    "adv_ratio",
    "exclamation_freq",
    "question_freq",
    "comma_freq",
    "avg_sentence_length"
]

# Our target variable 
target = "categorie"


In [5]:

X_stylo = df[stylometric_columns].values

# Correct method
X_tfidf = tfidf.transform(df["paragraph"])

X = hstack([
    X_tfidf,
    csr_matrix(X_stylo)
])

y = df[target]

In [6]:
# We then split our data 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [7]:
# We standardize our features 
scaler = StandardScaler(with_mean=False)  # IMPORTANT pour sparse matrix

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 4. Data Modeling 

### 4.0 Naive Model

In [8]:
y_pred_naive = [0]*len(y_test)

print("---Performances of the Naive Model---\n")
print("Accuracy:",metrics.accuracy_score(y_pred_naive, y_test))


---Performances of the Naive Model---

Accuracy: 0.912249443207127


### 4.1 Linear Regression

In [9]:
# Create model instance
lin_reg = LogisticRegression(max_iter=1000, class_weight="balanced")

# Fit the model
lin_reg.fit(X_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [10]:
# Use the model to predict and then evaluate its performances 

# y_pred for test set 
y_pred_float = lin_reg.predict(X_test)
y_pred = [1 if x>=0.5 else 0 for x in y_pred_float]

# y_pred for train set 
y_pred_train_float = lin_reg.predict(X_train)
y_pred_train = [1 if x>=0.5 else 0 for x in y_pred_train_float]

In [11]:
# Print the evaluated performances 

print("---Performances of the Linear regression on train data---\n")
print("Accuracy:",metrics.accuracy_score(y_pred_train, y_train))
print("F1-Score:",metrics.f1_score(y_pred_train, y_train))
print("Recall",metrics.recall_score(y_pred_train, y_train))

print("\n\n---Performances of the Linear regression on test data---\n")

print("Accuracy:",metrics.accuracy_score(y_pred, y_test))
print("F1-Score:",metrics.f1_score(y_pred, y_test))
print("Recall",metrics.recall_score(y_pred, y_test))

---Performances of the Linear regression on train data---

Accuracy: 0.9989977728285078
F1-Score: 0.9943145925457991
Recall 0.9886934673366834


---Performances of the Linear regression on test data---

Accuracy: 0.911804008908686
F1-Score: 0.56
Recall 0.4980237154150198


### 4.2 Random Forest 

In [12]:
rf = RandomForestClassifier(class_weight="balanced")
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_train_rf = rf.predict(X_train)

In [13]:
# Print the evaluated performances 

print("---Performances of the Random Forest on train data---\n")
print("Accuracy:",metrics.accuracy_score(y_pred_train_rf, y_train))
print("F1-Score:",metrics.f1_score(y_pred_train_rf, y_train))
print("Recall",metrics.recall_score(y_pred_train_rf, y_train))

print("\n\n---Performances of the Random Forest on test data---\n")

print("Accuracy:",metrics.accuracy_score(y_pred_rf, y_test))
print("F1-Score:",metrics.f1_score(y_pred_rf, y_test))
print("Recall",metrics.recall_score(y_pred_rf, y_test))

---Performances of the Random Forest on train data---

Accuracy: 0.9998886414253898
F1-Score: 0.9993642720915448
Recall 1.0


---Performances of the Random Forest on test data---

Accuracy: 0.9256124721603564
F1-Score: 0.2643171806167401
Recall 1.0


### 4.3 SVM 

In [14]:

#Create a svm Classifier
model_svm = svm.SVC(kernel='poly', class_weight="balanced") # Polynomial Kernel

#Train the model using the training sets
model_svm.fit(X_train, y_train)

#Predict the response for test dataset
y_pred_svm_train = model_svm.predict(X_train)
y_pred_svm = model_svm.predict(X_test)

In [15]:

# Print the evaluated performances 

print("---Performances of the SVM on train data---\n")
print("Accuracy:",metrics.accuracy_score(y_pred_svm_train, y_train))
print("F1-Score:",metrics.f1_score(y_pred_svm_train, y_train))
print("Recall",metrics.recall_score(y_pred_svm_train, y_train))

print("\n\n---Performances of the SVM on test data---\n")

print("Accuracy:",metrics.accuracy_score(y_pred_svm, y_test))
print("F1-Score:",metrics.f1_score(y_pred_svm, y_test))
print("Recall",metrics.recall_score(y_pred_svm, y_test))



---Performances of the SVM on train data---

Accuracy: 0.9997772828507795
F1-Score: 0.998730964467005
Recall 0.9974651457541192


---Performances of the SVM on test data---

Accuracy: 0.9447661469933185
F1-Score: 0.5921052631578947
Recall 0.8411214953271028


## 5. Models Download 

In [16]:
def download_model(model, results_dir: str, filename: str = "model.pkl"):
    """
    Save a trained model to disk using joblib.
    """
    
    model_path = f"{results_dir}//{filename}"
    joblib.dump(model, model_path)

    print(f"✅ Model saved to {model_path}")


In [17]:
download_model(lin_reg, "metrics//models", "lin_reg.pkl")
download_model(rf, "metrics//models", "rf.pkl")
download_model(model_svm, "metrics//models", "svm.pkl")

✅ Model saved to metrics//models//lin_reg.pkl
✅ Model saved to metrics//models//rf.pkl
✅ Model saved to metrics//models//svm.pkl
